# Backward Chaining in First Order Logic

Backward chaining is a form of logical inference that starts from a **goal (hypothesis)** and works backwards to find supporting **facts (evidence)**.

This method is best for when you have a small set of hypothesis and a large set of premises. If the converse is true, forward chaining might be a better approach.

It works like a reverse DFS: starting from the goal and recursively breaking it down into sub-goals until we reach base facts. Look at the printed example in the cell below for a better visualization.

To test this out for yourself: run the cell below. You can also change "print(backchain_to_goal_tree(zookeeper_rules, "opus is a penguin"))" to a different set of rules from a different knowledgebase (formatted like zookeeper_rules in data.py) and run your own hypothesis.

In [5]:
# Import additional methods for backchaining
from production import AND, OR, NOT, PASS, FAIL, IF, THEN, match, populate, simplify, variables
from data import zookeeper_rules

def backchain_to_goal_tree(rules, hypothesis, depth = 0):
    """
    Takes a hypothesis (string) and a list of rules (list
    of IF objects), returning an AND/OR tree representing the
    backchain of possible statements we may need to test
    to determine if this hypothesis is reachable or not.

    This method should return an AND/OR tree, that is, an
    AND or OR object, whose constituents are the subgoals that
    need to be tested. The leaves of this tree should be strings
    (possibly with unbound variables), *not* AND or OR objects.
    Make sure to use simplify(...) to flatten trees where appropriate.
    """
    ans = [hypothesis]

    #-- visual--#
    i = "   " * depth 
    print(f'{i}curr: {hypothesis}')
    #-----------#

    for rule in rules:

        m = match(rule.consequent(), hypothesis)
        if m is None:
            continue 

        # instantiate 
        antecedent = populate(rule.antecedent(), m)

        # check instances
        if isinstance(antecedent, str):
            ans.append(backchain_to_goal_tree(rules, antecedent, depth+1))

        elif isinstance(antecedent, AND):
            new_conditions = [backchain_to_goal_tree(rules, c, depth+1) for c in antecedent]
            ans.append(AND(*new_conditions))

        elif isinstance(antecedent, OR):
            new_conditions = [backchain_to_goal_tree(rules, c, depth+1) for c in antecedent]
            ans.append(OR(*new_conditions))

    #- visual every return -#
    print (f'{i}-> {ans}')
    # ----------------------#

    return simplify(OR(*ans))

print(backchain_to_goal_tree(zookeeper_rules, "opus is a penguin"))

curr: opus is a penguin
   curr: opus is a bird
      curr: opus has feathers
      -> ['opus has feathers']
      curr: opus flies
      -> ['opus flies']
      curr: opus lays eggs
      -> ['opus lays eggs']
   -> ['opus is a bird', AND('opus has feathers'), AND('opus flies', 'opus lays eggs')]
   curr: opus does not fly
   -> ['opus does not fly']
   curr: opus swims
   -> ['opus swims']
   curr: opus has black and white color
   -> ['opus has black and white color']
-> ['opus is a penguin', AND(OR('opus is a bird', 'opus has feathers', AND('opus flies', 'opus lays eggs')), 'opus does not fly', 'opus swims', 'opus has black and white color')]
OR('opus is a penguin', AND(OR('opus is a bird', 'opus has feathers', AND('opus flies', 'opus lays eggs')), 'opus does not fly', 'opus swims', 'opus has black and white color'))
